# DR-XAI: train the specified model on a Colab GPU

Before running: **Runtime -> Change runtime type -> T4 GPU**, and upload `dr_xai_colab.zip` to the top level of your Google Drive (My Drive). Run the cells in order. Checkpoints are written to `My Drive/dr_xai_checkpoints`, so a disconnect loses at most one epoch.

In [ ]:
# 1. Mount Drive and unpack the bundle (code + preprocessed images + locked split)
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/dr-xai && mkdir -p /content/dr-xai
!unzip -q -o /content/drive/MyDrive/dr_xai_colab.zip -d /content/dr-xai
%cd /content/dr-xai
!ls && ls data/processed/dr512/aptos2019/images | wc -l   # expect 3662

In [ ]:
# 2. Packages and GPU check
!pip install -q timm==1.0.30 albumentations==2.0.8 pandas==3.0.6 grad-cam==1.5.7
import os, sys, torch
os.environ['PYTHONPATH'] = 'src'
os.environ['PYTHONHASHSEED'] = '42'
CKPT = '/content/drive/MyDrive/dr_xai_checkpoints'
os.makedirs(CKPT, exist_ok=True)
print(sys.version)
print(torch.__version__, torch.cuda.get_device_name(0))   # must print a GPU name

In [ ]:
# 3. Train the specified model: maxvit_tiny_tf_512, 512 px, up to 30 epochs, seed 42
#    The last line printed is 'run id: <RUN_ID>'. Copy it into the next cell.
#    Out of memory? add:  train.batch_size=8 train.grad_accum=4
!python -m xaidr.cli train paths.checkpoints=$CKPT train.num_workers=2

In [ ]:
# 3b. ONLY if the session disconnected during cell 3: rerun cells 1-2, then this one
RUN = 'PASTE_RUN_ID_HERE'
# !python -m xaidr.cli train paths.checkpoints=$CKPT train.num_workers=2 --resume $RUN

In [ ]:
# 4. Evaluate once on the held-out test split (writes Table 4 and Table 6)
RUN = 'PASTE_RUN_ID_HERE'
!python -m xaidr.cli evaluate --run $RUN paths.checkpoints=$CKPT train.num_workers=2
!mkdir -p $CKPT/results_$RUN && cp -r results/* $CKPT/results_$RUN/
!ls -la $CKPT/$RUN $CKPT/results_$RUN

## Optional extra runs (each is another full training)

Run these only if there is GPU time left. Each prints its own run id.

- **Ablation arm, no oversampling:** needed for the Section 6.4 comparison.
- **Seeds 43 and 44:** needed if the paper keeps the 'explanation stability' claim.

In [ ]:
# 5. Optional: ablation arm without minority oversampling
# !python -m xaidr.cli train paths.checkpoints=$CKPT train.num_workers=2 minority.mode=none

In [ ]:
# 6. Optional: two more seeds
# !python -m xaidr.cli train paths.checkpoints=$CKPT train.num_workers=2 seed=43
# !python -m xaidr.cli train paths.checkpoints=$CKPT train.num_workers=2 seed=44

## When finished

In Google Drive open `dr_xai_checkpoints`. Download each run folder (it contains `best.pt`, `run.json`, `threshold.json`, `metrics.jsonl`, `config.yaml`, prediction CSVs) and the matching `results_<RUN_ID>` folder. Put the run folders into the project's `checkpoints/` folder on the laptop, then tell the Engineer the run ids.